In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE catalog1.gold_layer.dim_policy AS
SELECT 
    policy_id,
    policy_name,
    yearly_limit,
    icu_daily_limit,
    daily_room_limit,
    max_days,
    waiting_period,
    icu_covered
FROM catalog1.gold_layer.gold_policy_table
""")

In [0]:
%sql
select * from catalog1.gold_layer.dim_policy

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE catalog1.gold_layer.dim_patient AS
SELECT DISTINCT
    patient_id,
    policy_name
FROM catalog1.gold_layer.gold_claims_table
""")

In [0]:
%sql
select * from catalog1.gold_layer.dim_patient

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE catalog1.gold_layer.dim_procedure AS
SELECT DISTINCT
    patient_id,
    procedure,
    diagnosis
FROM catalog1.gold_layer.gold_claims_table
""")

In [0]:
%sql
select * from catalog1.gold_layer.dim_procedure

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE catalog1.gold_layer.dim_date AS
SELECT DISTINCT
    admission_date,
    discharge_date,
    to_date(admission_date, 'dd-MMM-yyyy')  AS admission_date_formatted,
    to_date(discharge_date, 'dd-MMM-yyyy')  AS discharge_date_formatted,
    year(to_date(admission_date, 'dd-MMM-yyyy'))  AS year,
    month(to_date(admission_date, 'dd-MMM-yyyy')) AS month,
    day(to_date(admission_date, 'dd-MMM-yyyy'))   AS day
FROM catalog1.gold_layer.gold_claims_table
""")

In [0]:
%sql
select * from catalog1.gold_layer.dim_date

In [0]:


spark.sql("""
CREATE OR REPLACE TABLE catalog1.gold_layer.fact_claims AS
SELECT
    c.claim_id,
    c.patient_id,
    c.policy_name,
    c.procedure,
    c.diagnosis,
    c.procedure_cost,
    c.total_amount,
    c.admission_date,
    c.discharge_date,
    f.Average_Cost,
    f.allowed_limit,
    f.price_difference,
    f.overpricing_percentage,
    f.overpricing_fraud,
    f.fraud_reason
FROM catalog1.gold_layer.gold_claims_table c
LEFT JOIN catalog1.gold_layer.gold_fraud_table f
    ON f.claim_id = c.claim_id
""")
print("✅ fact_claims created")

In [0]:
%sql
select * from catalog1.gold_layer.fact_claims